# Notebook 05 — Baseline ML Models

Trains XGBoost and LightGBM in two variants:
- Without graph features (pure trip-level)
- With graph features (centrality + corridor stats)

Generates SHAP importance plots.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import shap
import matplotlib.pyplot as plt
from src.models.baseline import run, evaluate, BASE_FEATURES, GRAPH_FEATURES
print('Baseline modules loaded')

In [ ]:
# Run baseline (no graph features)
results_no_graph = run(use_graph_features=False)
print('\n=== WITHOUT GRAPH FEATURES ===')
for r in results_no_graph:
    print(r)

In [ ]:
# Run with graph features
results_graph = run(use_graph_features=True)
print('\n=== WITH GRAPH FEATURES ===')
for r in results_graph:
    print(r)

In [ ]:
# Compare results
all_results = results_no_graph + results_graph
results_df = pd.DataFrame(all_results)
results_df.to_csv('data/processed/baseline_results.csv', index=False)
print(results_df[['model','MAE','within_15pct','MAPE']].to_string())

In [ ]:
# SHAP analysis
import pickle
try:
    with open('data/processed/models/XGBoost+Graph.pkl', 'rb') as f:
        model = pickle.load(f)
    df = pd.read_parquet('data/processed/features.parquet')
    from src.models.baseline import add_graph_features
    df = add_graph_features(df)
    available = [c for c in BASE_FEATURES + GRAPH_FEATURES if c in df.columns]
    X = df[available].fillna(0)
    X_sample = X.sample(min(2000, len(X)), random_state=42)
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, show=False, max_display=15)
    plt.tight_layout()
    plt.savefig('reports/05_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
except FileNotFoundError:
    print('Train model first by running the cells above.')